# COVID-19-Dashboard_NB_v9.20

## Init

In [3]:
%load_ext skip_kernel_extension
# Kernel Extensions: To skip a cell set %%skip True   

Skip kernel extension loaded!


## Debug

In [5]:
%%skip False # Debug: Show debug dataset in GUI

debug_show_dataset_in_tab1 = False
debug_show_dataset_in_tab2 = False
debug_show_dataset_in_tab3 = False

## Import core modules

In [7]:
%%skip False # Import: modules 

import os, sys
from pathlib import Path
from datetime import datetime

project_root = Path.cwd()
while not (project_root / 'config.py').exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, project_root.as_posix())

SRC_DIR = project_root / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from skip_kernel_extension import skip  # loads from notebooks working dir
from config import SRC_DIR, CSV_DIR, PROV_SHAPEFILE, MUN_SHAPEFILE, RWZI_FILE, RNA_FILE, SHAPEFILES_DIR, EXPORTS_DIR, UTILS_DIR
from data_loader import load_province_shapefile

from data_service import (
    get_province_heatmap_data,
    get_riool_heatmap_data,
    get_municipality_heatmap_data,
    get_prepared_riool_dataset,
    get_metric_mapping_tab1,
    get_metric_mapping_tab2,
    get_metric_mapping_tab3,
    get_province_riool_heatmap_data,
    get_municipality_riool_heatmap_data
)

from covid_dashboard_presenter import (
    # plot_heatmap,
    plot_province_heatmap,
    plot_municipality_heatmap,
    plot_municipality_heatmap_riool,
    plot_province_heatmap_riool
)

from src.utils.mapping_utils import get_metric_mapping_tab3
from src.utils.debug_utils import print_clean_debug_info
from src.utils.notebook_setup import init_notebook_environment
from src.utils.export_utils import create_export_widget

# init_notebook_environment()

## Import other modules

In [9]:
%%skip False # Import: other modules

from IPython.display import display, clear_output
import pandas as pd
import geopandas as gpd
import ipywidgets as widgets
import matplotlib.pyplot as plt

## Dataframes

In [11]:
%%skip False  # Dataframes: Covid-19 Central Dataset

from data_service import get_prepared_covid_dataset
df = get_prepared_covid_dataset()

# print("Kolommen:", df.columns.tolist())

## Available years

In [13]:
%%skip False  # Available Years: Dashboard default settings for scope and start

from data_service import get_available_years
available_years, default_year = get_available_years(df)

# print("Unieke jaartallen (Year):", sorted(df['Year'].dropna().unique()))
# print("Default Year:", default_year)

## Export state

In [15]:
%%skip False  # Export state for controls: Dashboard default settings

# Timestamp for tabX_get_plot_png
def ts():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

# --- Export state (per tab) ---
tab1_state = {"df_filtered": pd.DataFrame(), "df_grouped": pd.DataFrame(), "filters": {}}
tab2_state = {"gdf": None, "column": None, "filters": {}}
tab3_state = {"gdf": None, "column": None, "filters": {}}

def _pick_first_existing(columns, candidates):
    return next((c for c in candidates if c in columns), None)

# Tabs

## Tab 1

### Tab 1 Controls

In [17]:
%%skip False  # Tab1 Controls

# Col 1: Year selection
year_dropdown_1 = widgets.Dropdown(description='Year', options=available_years, value=default_year)

# Col 2: Region selection
province_dropdown_1 = widgets.Dropdown(
    description='Region',
    options=['Netherlands'] + sorted(df['Province_merged'].dropna().unique()),
    value='Netherlands'
)
months_checkbox_1 = widgets.Checkbox(value=True, description='Months')
municipalities_checkbox_1 = widgets.Checkbox(value=False, description='Municipalities')
province_chart_checkbox_1 = widgets.Checkbox(value=False, description='Provinces')

# Col 3: Export and metrics
def tab1_get_small():
    df_g = tab1_state.get("df_grouped", pd.DataFrame())
    if df_g is None or df_g.empty:
        return pd.DataFrame()
    return df_g.reset_index().head(25)

def tab1_get_full():
    return tab1_state.get("df_filtered", pd.DataFrame())

def tab1_get_meta():
    return tab1_state.get("filters", {})
export_widget_1 = create_export_widget(
    get_small=tab1_get_small,
    get_full=tab1_get_full,
    get_meta=tab1_get_meta,
    label="Tab1_export",
    width="300px",
)
# placeholder_df_1 = pd.DataFrame()
# export_widget_1 = create_export_widget(data_small=placeholder_df_1, data_full=placeholder_df_1, label="Tab1_export")

total_reported_checkbox_1 = widgets.Checkbox(value=True, description='Total reported')
hospital_admission_checkbox_1 = widgets.Checkbox(value=False, description='Hospital admissions')
deceased_checkbox_1 = widgets.Checkbox(value=False, description='Deceased')

tab1_col1 = widgets.VBox([
    year_dropdown_1
])

tab1_col2 = widgets.VBox([
    province_dropdown_1,
    months_checkbox_1,
    province_chart_checkbox_1,
    municipalities_checkbox_1
])

tab1_col3 = widgets.VBox([
    total_reported_checkbox_1,
    hospital_admission_checkbox_1,
    deceased_checkbox_1,
])

### Tab 1 Fliplogic 

In [19]:
%%skip False  # Tab 1 Fliplogic for Region checkboxes

# Dynamisch tonen/verbergen van checkboxes obv Region
def update_checkbox_visibility(change):
    if change['new'] == 'Netherlands':
        municipalities_checkbox_1.layout.display = 'none'
        municipalities_checkbox_1.value = False  # logisch uitschakelen
        months_checkbox_1.value = True           # altijd actief
        province_chart_checkbox_1.layout.display = ''
    else:
        province_chart_checkbox_1.layout.display = 'none'
        province_chart_checkbox_1.value = False  # logisch uitschakelen
        months_checkbox_1.value = True           # altijd actief
        municipalities_checkbox_1.layout.display = ''

# Aanroepen bij interactie
province_dropdown_1.observe(update_checkbox_visibility, names='value')

# Initiale zichtbaarheid instellen (default = Netherlands)
update_checkbox_visibility({'new': province_dropdown_1.value})

# Fliplogica voor de zichtbare checkboxen (2 tegelijk zichtbaar)
def on_visible_checkbox_change(change):
    if change['owner'].description == "Months" and change['new'] is True:
        if province_dropdown_1.value == "Netherlands":
            province_chart_checkbox_1.value = False
        else:
            municipalities_checkbox_1.value = False

    elif change['owner'].description in ["Provinces", "Municipalities"] and change['new'] is True:
        months_checkbox_1.value = False

months_checkbox_1.observe(on_visible_checkbox_change, names='value')
province_chart_checkbox_1.observe(on_visible_checkbox_change, names='value')
municipalities_checkbox_1.observe(on_visible_checkbox_change, names='value')

### Tab 1 Plot

In [21]:
%%skip False  # Tab 1 Plot and debug

plot_output_1 = widgets.Output()
debug_output_1 = widgets.Output()

def update_tab1_plot(year, total_reported, hospital_admission, deceased, province, municipalities, months, province_chart):

    from covid_dashboard_presenter import plot_covid

    # Metrics selecteren
    columns = []
    if total_reported:
        columns.append("Total_reported")
    if hospital_admission:
        columns.append("Hospital_admission")
    if deceased:
        columns.append("Deceased")

    year = str(year)
    df_filtered = df[df["Year"] == int(year)]

    if province != "Netherlands":
        df_filtered = df_filtered[df_filtered["Province_merged"] == province]
        group_col = "Municipality_name_merged"
    else:
        group_col = "Province_merged"

    df_grouped = df_filtered.groupby(group_col)[columns].sum(numeric_only=True).sort_values(by=columns[0], ascending=False) if columns else pd.DataFrame()

    # State update Export controls
    tab1_state["df_filtered"] = df_filtered.copy()
    tab1_state["df_grouped"] = df_grouped.copy() if df_grouped is not None else pd.DataFrame()
    tab1_state["filters"] = {
        "Tab": "Tab1",
        "Year": year,
        "Province": province,
        "Municipalities": municipalities,
        "Months": months,
        "Metrics": ", ".join(columns) if columns else "(none)",
    }
   
    with plot_output_1:
        clear_output(wait=True)
        plot_covid(
            df=df_filtered,
            year=year,
            total_reported=total_reported,
            hospital_admission=hospital_admission,
            deceased=deceased,
            province=province,
            municipalities=municipalities,
            months=months
        )     
        # print(df_filtered.columns.tolist())
        # print(df_filtered.head(3))

        # # Debug info graphical file
        # print("DEBUG fignums:", plt.get_fignums())
        # if plt.get_fignums():
        #     fig = plt.figure(plt.get_fignums()[-1])
        #     print("DEBUG axes:", len(fig.axes))
        
        # Save current plot as PNG for PDF export
        plot_path = Path(EXPORTS_DIR) / "Tab1_current_plot.png"
        plt.gcf().savefig(plot_path, dpi=150, bbox_inches="tight")
        tab1_state["plot_path"] = str(plot_path)
    
    with debug_output_1:
        clear_output(wait=True)
        
        if debug_show_dataset_in_tab1:
            for metric in columns:
                filters = {
                    "Year": year,
                    "Province": province,
                    "Municipalities": municipalities,
                    "Metric": metric
                }
                print_clean_debug_info(df_filtered, df_grouped.reset_index(), group_col, metric, f"{metric} in {year}", filters)
            print("✅ End debug for Tab1.")

### Tab 1 Outputzone

In [23]:
%%skip False  # Tab 1 GUI-built outputzone

ui_1 = widgets.VBox([
    widgets.HBox([tab1_col1, tab1_col2, tab1_col3], layout=widgets.Layout(gap='40px')),
    plot_output_1,
    export_widget_1,
    debug_output_1
])

## Tab 2

### Tab 2 Controls

In [25]:
%%skip False  # Tab2 Controls

# Col 1: Year selection
year_dropdown_2 = widgets.Dropdown(description='Year', options=available_years, value=default_year)

# Col 2: Level selection
level_dropdown_2 = widgets.Dropdown(description='Region', options=['Provinces', 'Municipalities'], value='Provinces')

# Col 3: Export and metrics
def tab2_get_small():
    gdf = tab2_state.get("gdf")
    column = tab2_state.get("column")
    if gdf is None or column is None or getattr(gdf, "empty", True) or column not in gdf.columns:
        return pd.DataFrame()

    id_col = _pick_first_existing(
        gdf.columns,
        ["Municipality_name_merged", "Province_merged", "GM_NAAM", "NAAM", "MUN_NAME", "PROV_NAME"],
    )
    cols = [c for c in [id_col, column] if c is not None]
    df = gdf[cols].copy()
    df = df.sort_values(column, ascending=False).head(25)
    return df

def tab2_get_full():
    return tab2_state.get("gdf")

def tab2_get_meta():
    meta = dict(tab2_state.get("filters", {}))
    if "plot_path" in tab2_state:
        meta["plot_path"] = tab2_state["plot_path"]
    return meta    
    # return tab2_state.get("filters", {})

def tab2_get_plot_png():
    gdf = tab2_state.get("gdf")
    column = tab2_state.get("column")
    filters = tab2_state.get("filters", {})
    if gdf is None or column is None:
        return None

    region = filters.get("Region", "Provinces")
    plot_path = Path(EXPORTS_DIR) / f"Tab2_plot_{ts()}.png"

    if region == "Provinces":
        plot_province_heatmap(gdf, column=column, save_as=str(plot_path), dpi=150)
    else:
        plot_municipality_heatmap(gdf, column=column, save_as=str(plot_path), dpi=150)

    return str(plot_path)
export_widget_2 = create_export_widget(
    get_small=tab2_get_small,
    get_full=tab2_get_full,
    get_meta=tab2_get_meta,
    get_plot_png=tab2_get_plot_png,
    label="Tab2_export",
    width="300px",
)

# placeholder_df_2 = pd.DataFrame()
# export_widget_2 = create_export_widget(data_small=placeholder_df_2, data_full=placeholder_df_2, label="Tab2_export")

metric_dropdown_2 = widgets.Dropdown(description='Metrics', options=['Total reported', 'Hospital admissions', 'Deceased'], value='Total reported')

tab2_col1 = widgets.VBox([
    year_dropdown_2
])

tab2_col2 = widgets.VBox([
    level_dropdown_2
])

tab2_col3 = widgets.VBox([
    metric_dropdown_2,
])

### Tab 2 Fliplogic

### Tab 2 Plot

In [28]:
%%skip False  # Tab 2 Plot and debug

plot_output_2 = widgets.Output()
debug_output_2 = widgets.Output()

def update_tab2_plot(year, region, metric):
    mapping = get_metric_mapping_tab2()
    column = mapping[metric]

    gdf = None  # altijd definiëren, ook bij foutpad

    with plot_output_2:
        clear_output(wait=True)
        if region == "Provinces":
            gdf = get_province_heatmap_data(year, column)
            gdf.attrs["year_label"] = str(year)
            plot_province_heatmap(gdf, column=column)

            # Save current plot as PNG for PDF export
            plot_path = Path(EXPORTS_DIR) / "Tab2_current_plot.png"
            plt.gcf().savefig(plot_path, dpi=150, bbox_inches="tight")
            tab2_state["plot_path"] = str(plot_path)
            
        elif region == "Municipalities":
            gdf = get_municipality_heatmap_data(year, column)
            gdf.attrs["year_label"] = str(year)
            plot_municipality_heatmap(gdf, column=column)

            # Save current plot as PNG for PDF export
            plot_path = Path(EXPORTS_DIR) / "Tab2_current_plot.png"
            plt.gcf().savefig(plot_path, dpi=150, bbox_inches="tight")
            tab2_state["plot_path"] = str(plot_path)
            
        else:
            print(f"⚠️ Onbekende regioselectie: {region}")
            gdf = None


    # --- state update ---
    tab2_state["gdf"] = gdf
    tab2_state["column"] = column
    tab2_state["filters"] = {"Tab": "Tab2", "Year": year, "Region": region, "Metric": metric}

    with debug_output_2:
        clear_output(wait=True)
        if debug_show_dataset_in_tab2 and gdf is not None:

            # Geo quick checks
            try:
                crs = getattr(gdf, "crs", None)
                geom = getattr(gdf, "geometry", None)
                if crs is not None:
                    print(f"CRS: {crs}")
                if geom is not None:
                    print(f"Geom types: {list(geom.geom_type.value_counts().to_dict().keys())}")
            except Exception as e:
                print(f"⚠️ Geo-check faalde: {e}")
        
            # Column stats
            if column in gdf.columns:
                s = gdf[column]
                nan_cnt = int(s.isna().sum())
                print(f"{column}: NaN={nan_cnt:,} | min={s.min()} | max={s.max()}")
            else:
                print(f"⚠️ Kolom ontbreekt in gdf: {column}")
            
            # Zoek automatisch een geschikte kolom
            group_col = next(
                (col for col in ["Municipality_name_merged", "Province_merged", "GM_NAAM", "NAAM"] if col in gdf.columns),
                None
            )
            filters = {
                "Year": year,
                "Region": region,
                "Metric": metric
            }
            print_clean_debug_info(gdf, gdf, group_col, column, f"{column} in {year}", filters)
            print("✅ End debug for Tab2.")
            
        elif debug_show_dataset_in_tab2 and gdf is None:
            print("⚠️ Geen gdf beschikbaar (regio onbekend of data faalde).")

### Tab 2 Outputzone

In [30]:
%%skip False  # Tab 2 GUI-built outputzone

ui_2 = widgets.VBox([
    widgets.HBox([tab2_col1, tab2_col2, tab2_col3], layout=widgets.Layout(gap='40px')),
    plot_output_2,
    export_widget_2,
    debug_output_2
])

## Tab 3

### Tab 3 Controls

In [32]:
%%skip False  # Tab 3 Controls

# Col 1: Year selection
year_dropdown_3 = widgets.Dropdown(description='Year', options=available_years, value=default_year)

# Col 2: Level selection
level_dropdown_3 = widgets.Dropdown(description='Region', options=['Provinces', 'Municipalities'], value='Provinces')

# Col 3: Export and metrics
def tab3_get_small():
    gdf = tab3_state.get("gdf")
    column = tab3_state.get("column")
    if gdf is None or column is None or getattr(gdf, "empty", True) or column not in gdf.columns:
        return pd.DataFrame()

    id_col = _pick_first_existing(
        gdf.columns,
        ["Provincienaam", "gemeentenaam", "Province_merged", "Municipality_name_merged", "GM_NAAM", "NAAM"],
    )
    cols = [c for c in [id_col, column] if c is not None]
    df = gdf[cols].copy()
    df = df.sort_values(column, ascending=False).head(25)
    return df

def tab3_get_full():
    return tab3_state.get("gdf")

def tab3_get_meta():
    meta = dict(tab3_state.get("filters", {}))
    if "plot_path" in tab3_state:
        meta["plot_path"] = tab3_state["plot_path"]
    return meta
    # return tab3_state.get("filters", {})

def tab3_get_plot_png():
    gdf = tab3_state.get("gdf")
    column = tab3_state.get("column")
    filters = tab3_state.get("filters", {})
    if gdf is None or column is None:
        return None

    region = filters.get("Region", "Provinces")
    plot_path = Path(EXPORTS_DIR) / f"Tab3_plot_{ts()}.png"

    if region == "Provinces":
        plot_province_heatmap(gdf, column=column, save_as=str(plot_path), dpi=150)
    else:
        plot_municipality_heatmap(gdf, column=column, save_as=str(plot_path), dpi=150)

    return str(plot_path)
export_widget_3 = create_export_widget(
    get_small=tab3_get_small,
    get_full=tab3_get_full,
    get_meta=tab3_get_meta,
    get_plot_png=tab3_get_plot_png,
    label="Tab3_export",
    width="300px",
)

# placeholder_df_3 = pd.DataFrame()
# export_widget_3 = create_export_widget(data_small=placeholder_df_3, data_full=placeholder_df_3, label="Tab3_export")

#metric_dropdown_3 = widgets.Dropdown(description='Metrics', options=['RNA flow per 100k', 'Normalized'], value='RNA flow per 100k')
metric_dropdown_3 = widgets.Dropdown(description='Metrics', options=['RNA flow per 100k'], value='RNA flow per 100k')

tab3_col1 = widgets.VBox([
    year_dropdown_3
])

tab3_col2 = widgets.VBox([
    level_dropdown_3
])

tab3_col3 = widgets.VBox([
    metric_dropdown_3,
])

### Tab 3 Fliplogic

### Tab 3 Plot

In [35]:
%%skip False  # Tab 3 Plot

plot_output_3 = widgets.Output()
debug_output_3 = widgets.Output()

def update_tab3_plot(year, region, metric):

    mapping = get_metric_mapping_tab3()
    column = mapping[metric]

    gdf = None  # altijd definiëren

    with plot_output_3:
        clear_output(wait=True)
        if region == "Provinces":
            gdf = get_province_riool_heatmap_data(year, region, column)
            gdf.attrs["year_label"] = str(year)
            plot_province_heatmap_riool(gdf, region, column)

            # Save current plot as PNG for PDF export
            plot_path = Path(EXPORTS_DIR) / "Tab3_current_plot.png"
            plt.gcf().savefig(plot_path, dpi=150, bbox_inches="tight")
            tab3_state["plot_path"] = str(plot_path)
            
        elif region == "Municipalities":
            gdf = get_municipality_riool_heatmap_data(year, region, column)
            gdf.attrs["year_label"] = str(year)
            plot_municipality_heatmap_riool(gdf, region, column)

            # Save current plot as PNG for PDF export
            plot_path = Path(EXPORTS_DIR) / "Tab3_current_plot.png"
            plt.gcf().savefig(plot_path, dpi=150, bbox_inches="tight")
            tab3_state["plot_path"] = str(plot_path)
        else:
            print(f"⚠️ Onbekende regioselectie: {region}")
            gdf = None

    # --- state update ---
    tab3_state["gdf"] = gdf
    tab3_state["column"] = column
    tab3_state["filters"] = {"Tab": "Tab3", "Year": year, "Region": region, "Metric": metric}

    # --- debug (NIET meer binnen plot_output_3!) ---
    with debug_output_3:
        clear_output(wait=True)
        if debug_show_dataset_in_tab3 and gdf is not None:
            # (jouw bestaande debugcode)
            ...
            print("✅ End debug for Tab3.")
        elif debug_show_dataset_in_tab3 and gdf is None:
            print("⚠️ Geen gdf beschikbaar (regio onbekend of data faalde).")

### Tab 3 Outputzone

In [37]:
%%skip False  # Tab 3 Outputzone

ui_3 = widgets.VBox([
    widgets.HBox([tab3_col1, tab3_col2, tab3_col3], layout=widgets.Layout(gap='40px')),
    plot_output_3,
    export_widget_3,
    debug_output_3
])

## Tab 4

### Tab 4 Controls

In [39]:
%%skip False  # Tab 4 Controls

import ipywidgets as widgets
from IPython.display import display, clear_output
from data_writer import Saveframes, Savepoly, Save_xlsx

download_button = widgets.Button(description="Download files", layout=widgets.Layout(width="130px"))

### Tab 4 Fliplogic

In [41]:
%%skip False  # Tab 4 Fliplogic

out4 = widgets.Output(layout=widgets.Layout(width="800px", height="600px"))

def run_download(_):
    with out4:
        clear_output(wait=True)
        print("📦 DOWNLOAD CSV")
        Saveframes()
        print("\n📦 DOWNLOAD XLSX")
        Save_xlsx()
        print("\n📦 DOWNLOAD POLYGONES")
        Savepoly()
download_button.on_click(run_download)

### Tab 4 Plot

In [43]:
%%skip False  # Tab 4 Plot
col1 = widgets.VBox([
    widgets.HTML("<h4>RIVM Covid-19</h4>"),
    download_button
], layout=widgets.Layout(width="170px", padding="10px"))

col2 = widgets.VBox([
    widgets.HTML("<h4>Output</h4>"),
    out4
], layout=widgets.Layout(width="830px", padding="10px"))

### Tab 4 Outputzone

In [45]:
%%skip False  # Tab 4: GUI-built outputzone

ui_4 = widgets.VBox([
    widgets.HBox([col1, col2], layout=widgets.Layout(gap='40px'))
])

## Tab 5

### Tab 5 Controls

In [47]:
%%skip False  # Tab 5: Controls

import ipywidgets as widgets
from IPython.display import display, clear_output
from src.utils.package_check import check_required_packages, get_environment_info, show_widget_test_plot

diagnose_button = widgets.Button(description="Check Python", layout=widgets.Layout(width="130px"))

### Tab 5 Fliplogic

In [49]:
%%skip False  # Tab 5: Fliplogic modules

out5 = widgets.Output(layout=widgets.Layout(width="800px", height="600px"))

def run_environment_diagnostics(_):
    with out5:
        clear_output(wait=True)
        print("🧠 ENVIRONMENT INFO")
        info = get_environment_info()
        for k, v in info.items():
            print(f"{k}: {v}")
        print("\n📦 PACKAGE CHECK")
        check_required_packages()

diagnose_button.on_click(run_environment_diagnostics)

### Tab 5 Plot

In [51]:
%%skip False  # Tab 5: Plot diagnostic info

col1 = widgets.VBox([
    widgets.HTML("<h4>Environment</h4>"),
    diagnose_button
], layout=widgets.Layout(width="170px", padding="10px"))

col2 = widgets.VBox([
    widgets.HTML("<h4>Output</h4>"),
    out5
], layout=widgets.Layout(width="830px", padding="10px"))

### Tab 5 Outputzone

In [53]:
%%skip False  # Tab 5: GUI-built outputzone

ui_5 = widgets.VBox([
    widgets.HBox([col1, col2], layout=widgets.Layout(gap='40px'))
])

# GUI Builder

## GUI Collection

### Output containers

In [55]:
%%skip False  # Dashboard: Output containers for Tabs

if "out1" not in globals():
    out1 = widgets.Output()

if "out2" not in globals():
    out2 = widgets.Output()

if "out3" not in globals():
    out3 = widgets.Output()

if "out4" not in globals():
    out4 = widgets.Output()

if "out5" not in globals():
    out5 = widgets.Output()

### Interactive outputs

#### Tab 1 - Interactive output

In [57]:
%%skip False  # Tab 1: Interactive output

interactive_tab1 = widgets.interactive_output(update_tab1_plot, {
    'year': year_dropdown_1,
    'total_reported': total_reported_checkbox_1,
    'hospital_admission': hospital_admission_checkbox_1,
    'deceased': deceased_checkbox_1,
    'province': province_dropdown_1,
    'municipalities': municipalities_checkbox_1,
    'months': months_checkbox_1,
    'province_chart': province_chart_checkbox_1
})

#### Tab 2 - Interactive output

In [59]:
%%skip False  # Tab 2: Interactive output

interactive_tab2 = widgets.interactive_output(
    update_tab2_plot,
    {
        'year': year_dropdown_2,
        'region': level_dropdown_2,
        'metric': metric_dropdown_2
    }
)

#### Tab 3 - Interactive output

In [61]:
%%skip False  # Tab 3: Interactive output

interactive_tab3 = widgets.interactive_output(
    update_tab3_plot,
    {
        'year': year_dropdown_3,
        'region': level_dropdown_3,
        'metric': metric_dropdown_3
    }
)

#### Tab 4 - Interactive output

In [63]:
%%skip False  # Tab 4 Interactive output

interactive_tab4 = NotImplemented

#### Tab 5 - Interactive output

In [65]:
%%skip False  # Tab 5 Interactive output

interactive_tab5 = NotImplemented 

## GUI Placeholders

### Tab 1 - GUI Placeholder out1

In [67]:
%%skip False  # Tab 1 GUI Placeholder

with out1:
    clear_output(wait=True)

    # Zorg dat ui_1 bestaat (val terug op bouwen als nodig)
    if 'ui_1' not in globals():
        if 'plot_output_1' not in globals():
            plot_output_1 = widgets.Output()
        if 'debug_output_1' not in globals():
            debug_output_1 = widgets.Output()

        # Gebruik tab1_col1/2/3 als die bestaan, anders minimal
        if all(k in globals() for k in ['tab1_col1', 'tab1_col2', 'tab1_col3']):
            header = widgets.HBox([tab1_col1, tab1_col2, tab1_col3], layout=widgets.Layout(gap='40px'))
        else:
            header = widgets.HTML("<b>Tab1 UI not built yet</b>")

        # export_widget_1 onder plot indien beschikbaar
        body = [header, plot_output_1]
        if 'export_widget_1' in globals():
            body.append(export_widget_1)
        body.append(debug_output_1)

        ui_1 = widgets.VBox(body)

    if not hasattr(out1, "done"):
        display(ui_1)
        out1.done = True

    display(interactive_tab1)

### Tab 2 - GUI Placeholder out2

In [69]:
%%skip False  # Tab 2 GUI Placeholder

with out2:
    clear_output(wait=True)
    if not hasattr(out2, "done"):
        display(ui_2)
        out2.done = True
    display(interactive_tab2)

### Tab 3 - GUI Placeholder out2

In [71]:
%%skip False  # Tab 3 GUI Placeholder

with out3:
    clear_output(wait=True)
    if not hasattr(out3, "done"):
        display(ui_3)
        out3.done = True
    display(interactive_tab3)

### Tab 4 - GUI Placeholder out4

In [73]:
%%skip True  # Tab 4 GUI Placeholder

with out4:
    clear_output(wait=True)
    if not hasattr(out4, "done"):
        display(ui_4)
        out4.done = True
    display(interactive_tab4)

### Tab 5 - GUI Placeholder out5

In [75]:
%%skip True  # Tab 5 GUI Placeholder

with out5:
    clear_output(wait=True)
    if not hasattr(out5, "done"):
        display(ui_5)
        out5.done = True
    display(interactive_tab5)

# Display Dashboard

## Render Dashboard

In [77]:
import chardet
with open(CSV_DIR / 'RWZI-verzorgingsgebied-per-gemeente-2025.csv', 'rb') as f:
    print(chardet.detect(f.read(50000)))

{'encoding': 'utf-8', 'confidence': 0.99, 'language': ''}


In [78]:
%%skip False  # Render GUI Dashboard

tabs = widgets.Tab(children=[out1, out2, out3, ui_4, ui_5])
tabs.set_title(0, 'Covid-19 Barcharts')
tabs.set_title(1, 'Covid-19 Heatmap')
tabs.set_title(2, 'Sewer Heatmap')
tabs.set_title(3, 'Download Dataset')
tabs.set_title(4, 'Diagnostics')

display(tabs)